# Live 3D Human Pose Estimation with OpenVINO

This notebook demonstrates live 3D Human Pose Estimation with OpenVINO via a webcam. We utilize the model [human-pose-estimation-3d-0001](https://github.com/openvinotoolkit/open_model_zoo/tree/master/models/public/human-pose-estimation-3d-0001) from [Open Model Zoo](https://github.com/openvinotoolkit/open_model_zoo/). At the end of this notebook, you will see live inference results from your webcam (if available). Alternatively, you can also upload a video file to test out the algorithms.
**Make sure you have properly installed the [Jupyter extension](https://github.com/jupyter-widgets/pythreejs#jupyterlab) and been using JupyterLab to run the demo as suggested in the `README.md`**

> **NOTE**: _To use a webcam, you must run this Jupyter notebook on a computer with a webcam. If you run on a remote server, the webcam will not work. However, you can still do inference on a video file in the final step. This demo utilizes the Python interface in `Three.js` integrated with WebGL to process data from the model inference. These results are processed and displayed in the notebook._

_To ensure that the results are displayed correctly, run the code in a recommended browser on one of the following operating systems:_
_Ubuntu, Windows: Chrome_
_macOS: Safari_


#### Table of contents:

- [Prerequisites](#Prerequisites)
- [Imports](#Imports)
- [The model](#The-model)
    - [Download the model](#Download-the-model)
    - [Convert Model to OpenVINO IR format](#Convert-Model-to-OpenVINO-IR-format)
    - [Select inference device](#Select-inference-device)
    - [Load the model](#Load-the-model)
- [Processing](#Processing)
    - [Model Inference](#Model-Inference)
    - [Draw 2D Pose Overlays](#Draw-2D-Pose-Overlays)
    - [Main Processing Function](#Main-Processing-Function)
- [Run](#Run)


### Installation Instructions

This is a self-contained example that relies solely on its own code.

We recommend  running the notebook in a virtual environment. You only need a Jupyter server to start.
For details, please refer to [Installation Guide](https://github.com/openvinotoolkit/openvino_notebooks/blob/latest/README.md#-installation-guide).

Make sure your [Jupyter extension](https://github.com/jupyter-widgets/pythreejs#jupyterlab) is working properly.
To avoid errors that may arise from the version of the dependency package, it is recommended to use the **JupyterLab** instead of the Jupyter notebook to display image results.
```
- pip install --upgrade pip && pip install -r requirements.txt
- jupyter labextension install --no-build @jupyter-widgets/jupyterlab-manager
- jupyter labextension install --no-build jupyter-datawidgets/extension
- jupyter labextension install jupyter-threejs
- jupyter labextension list
```

You should see:
```
JupyterLab v...
  ...
    jupyterlab-datawidgets v... enabled OK
    @jupyter-widgets/jupyterlab-manager v... enabled OK
    jupyter-threejs v... enabled OK
```

<img referrerpolicy="no-referrer-when-downgrade" src="https://static.scarf.sh/a.png?x-pxid=5b5a4db0-7875-4bfb-bdbd-01698b5b1a77&file=notebooks/3D-pose-estimation-webcam/3D-pose-estimation.ipynb" />


## Prerequisites
[back to top ⬆️](#Table-of-contents:)

**The `pythreejs` extension may not display properly when using a Jupyter Notebook release. Therefore, it is recommended to use Jupyter Lab instead.**

In [1]:
import platform

if platform.system() == "Darwin":
    %pip install -q "numpy<2.0.0"

%pip install pythreejs "openvino>=2024.4.0" "opencv-python" "torch" "tqdm" --extra-index-url https://download.pytorch.org/whl/cpu

Looking in indexes: https://pypi.org/simple, https://download.pytorch.org/whl/cpu
Note: you may need to restart the kernel to use updated packages.


## Imports
[back to top ⬆️](#Table-of-contents:)


In [2]:
import collections
import time
from pathlib import Path

import cv2
import ipywidgets as widgets
import numpy as np
from IPython.display import clear_output, display
import openvino as ov

# Fetch `notebook_utils` module
import requests

if not Path("notebook_utils.py").exists():
    r = requests.get(
        url="https://raw.githubusercontent.com/openvinotoolkit/openvino_notebooks/latest/utils/notebook_utils.py",
    )
    with open("notebook_utils.py", "w") as f:
        f.write(r.text)

if not Path("engine3js.py").exists():
    r = requests.get(
        url="https://raw.githubusercontent.com/openvinotoolkit/openvino_notebooks/latest/utils/engine3js.py",
    )
    with open("engine3js.py", "w") as f:
        f.write(r.text)

import notebook_utils as utils
import engine3js as engine

# Read more about telemetry collection at https://github.com/openvinotoolkit/openvino_notebooks?tab=readme-ov-file#-telemetry
from notebook_utils import collect_telemetry

collect_telemetry("3D-pose-estimation.ipynb")

## The model
[back to top ⬆️](#Table-of-contents:)

### Download the model
[back to top ⬆️](#Table-of-contents:)

In [11]:
from notebook_utils import download_file
import tarfile


# directory where model will be downloaded
base_model_dir = Path("model")

if not base_model_dir.exists():
    download_file(
        "https://storage.openvinotoolkit.org/repositories/open_model_zoo/public/2022.1/human-pose-estimation-3d-0001/human-pose-estimation-3d.tar.gz",
        directory=base_model_dir,
    )

ckpt_file = base_model_dir / "human-pose-estimation-3d-0001.pth"

if not ckpt_file.exists():
    with tarfile.open(base_model_dir / "human-pose-estimation-3d.tar.gz") as f:
        f.extractall(base_model_dir)

### Convert Model to OpenVINO IR format
[back to top ⬆️](#Table-of-contents:)

In [12]:
import torch

ov_model_path = Path(base_model_dir) / "human-pose-estimation-3d-0001.xml"

if not ov_model_path.exists():
    from model.model import PoseEstimationWithMobileNet

    pose_estimation_model = PoseEstimationWithMobileNet(is_convertible_by_mo=True)
    pose_estimation_model.load_state_dict(torch.load(ckpt_file, map_location="cpu"))
    pose_estimation_model.eval()

    with torch.no_grad():
        ov_model = ov.convert_model(pose_estimation_model, example_input=torch.zeros([1, 3, 256, 448]), input=[1, 3, 256, 448])
        ov.save_model(ov_model, ov_model_path)

### Select inference device
[back to top ⬆️](#Table-of-contents:)

select device from dropdown list for running inference using OpenVINO

In [13]:
device = utils.device_widget()

device

Dropdown(description='Device:', index=3, options=('CPU', 'GPU', 'NPU', 'AUTO'), value='AUTO')

### Load the model
[back to top ⬆️](#Table-of-contents:)

Converted models are located in a fixed structure, which indicates vendor, model name and precision.

First, initialize the inference engine, OpenVINO Runtime. Then, read the network architecture and model weights from the `.bin` and `.xml` files to compile for the desired device. An inference request is then created to infer the compiled model.

In [14]:
# initialize inference engine
core = ov.Core()
# read the network and corresponding weights from file
model = core.read_model(ov_model_path)
# load the model on the specified device
compiled_model = core.compile_model(model=model, device_name=device.value)

## Processing
[back to top ⬆️](#Table-of-contents:)

### Model Inference
[back to top ⬆️](#Table-of-contents:)

Frames captured from video files or the live webcam are used as the input for the 3D model. This is how you obtain the output heat maps, PAF (part affinity fields) and features.

In [15]:
def model_infer(scaled_img, stride):
    """
    Run model inference on the input image

    Parameters:
        scaled_img: resized image according to the input size of the model
        stride: int, the stride of the window
    """

    # Remove excess space from the picture
    img = scaled_img[
        0 : scaled_img.shape[0] - (scaled_img.shape[0] % stride),
        0 : scaled_img.shape[1] - (scaled_img.shape[1] % stride),
    ]

    mean_value = 128.0
    scale_value = 255.0

    img = (img - mean_value) / scale_value

    img = np.transpose(img, (2, 0, 1))[None,]
    result = compiled_model(img)
    # Get the results
    results = (result[0][0], result[1][0], result[2][0])

    return results

### Draw 2D Pose Overlays
[back to top ⬆️](#Table-of-contents:)

We need to define some connections between the joints in advance, so that we can draw the structure of the human body in the resulting image after obtaining the inference results.
Joints are drawn as circles and limbs are drawn as lines. The code is based on the [3D Human Pose Estimation Demo](https://github.com/openvinotoolkit/open_model_zoo/tree/master/demos/human_pose_estimation_3d_demo/python) from Open Model Zoo.

In [16]:
# 3D edge index array
body_edges = np.array(
    [
        [0, 1],
        [0, 9],
        [9, 10],
        [10, 11],  # neck - r_shoulder - r_elbow - r_wrist
        [0, 3],
        [3, 4],
        [4, 5],  # neck - l_shoulder - l_elbow - l_wrist
        [1, 15],
        [15, 16],  # nose - l_eye - l_ear
        [1, 17],
        [17, 18],  # nose - r_eye - r_ear
        [0, 6],
        [6, 7],
        [7, 8],  # neck - l_hip - l_knee - l_ankle
        [0, 12],
        [12, 13],
        [13, 14],  # neck - r_hip - r_knee - r_ankle
    ]
)


body_edges_2d = np.array(
    [
        [0, 1],  # neck - nose
        [1, 16],
        [16, 18],  # nose - l_eye - l_ear
        [1, 15],
        [15, 17],  # nose - r_eye - r_ear
        [0, 3],
        [3, 4],
        [4, 5],  # neck - l_shoulder - l_elbow - l_wrist
        [0, 9],
        [9, 10],
        [10, 11],  # neck - r_shoulder - r_elbow - r_wrist
        [0, 6],
        [6, 7],
        [7, 8],  # neck - l_hip - l_knee - l_ankle
        [0, 12],
        [12, 13],
        [13, 14],  # neck - r_hip - r_knee - r_ankle
    ]
)


def draw_poses(frame, poses_2d, scaled_img, use_popup):
    """
    Draw 2D pose overlays on the image to visualize estimated poses.
    Joints are drawn as circles and limbs are drawn as lines.

    :param frame: the input image
    :param poses_2d: array of human joint pairs
    """
    for pose in poses_2d:
        pose = np.array(pose[0:-1]).reshape((-1, 3)).transpose()
        was_found = pose[2] > 0

        pose[0], pose[1] = (
            pose[0] * frame.shape[1] / scaled_img.shape[1],
            pose[1] * frame.shape[0] / scaled_img.shape[0],
        )

        # Draw joints.
        for edge in body_edges_2d:
            if was_found[edge[0]] and was_found[edge[1]]:
                cv2.line(
                    frame,
                    tuple(pose[0:2, edge[0]].astype(np.int32)),
                    tuple(pose[0:2, edge[1]].astype(np.int32)),
                    (255, 255, 0),
                    4,
                    cv2.LINE_AA,
                )
        # Draw limbs.
        for kpt_id in range(pose.shape[1]):
            if pose[2, kpt_id] != -1:
                cv2.circle(
                    frame,
                    tuple(pose[0:2, kpt_id].astype(np.int32)),
                    3,
                    (0, 255, 255),
                    -1,
                    cv2.LINE_AA,
                )

    return frame

### Main Processing Function
[back to top ⬆️](#Table-of-contents:)

Run 3D pose estimation on the specified source. It could be either a webcam feed or a video file.

In [99]:
def run_pose_estimation(source=0, flip=False, use_popup=False, skip_frames=0, destination=None, verbose=False, annotate=True, exportPose=False):
    """
    2D image as input, using OpenVINO as inference backend,
    get joints 3D coordinates, and draw 3D human skeleton in the scene

    :param source:      The webcam number to feed the video stream with primary webcam set to "0", or the video path.
    :param flip:        To be used by VideoPlayer function for flipping capture image.
    :param use_popup:   False for showing encoded frames over this notebook, True for creating a popup window.
    :param skip_frames: Number of frames to skip at the beginning of the video.
    """

    focal_length = -1  # default
    stride = 8
    player = None
    skeleton_set = None

    frameWidth = None
    frameHeight = None

    exportVideo = False
    videoOut = None
    
    poseFile = None

    posePrinted = False

    try:
        # create video player to play with target fps  video_path
        # get the frame from camera
        # You can skip first N frames to fast forward video. change 'skip_first_frames'
        player = utils.VideoPlayer(source, flip=flip, fps=30, skip_first_frames=skip_frames)
        # start capturing
        player.start()

        input_image = player.next()
        
        # Get Image Size
        if frameWidth is None or frameHeight is None:
            frameWidth = input_image.shape[1]
            frameHeight = input_image.shape[0]
            if verbose:
                print("Frame Width = {}".format(frameWidth))
                print("Frame Height = {}".format(frameHeight))
            
        # set the window size
        resize_scale = 450 / input_image.shape[1]
        windows_width = int(input_image.shape[1] * resize_scale)
        windows_height = int(input_image.shape[0] * resize_scale)

        # Export output
        if destination is not None and videoOut is None and not exportPose:
            exportVideo = True
            if verbose:
                print("Creating Video Writer")
            fourcc = cv2.VideoWriter_fourcc(*"XVID")
            videoOut = cv2.VideoWriter(destination, fourcc, 30.0, (frameWidth, frameHeight))
            # videoOut = cv2.VideoWriter(destination, fourcc, 30.0, (windows_width, windows_height))
            if verbose:
                print("Created Video Writer pointing to location: {}".format(destination))

        if exportPose and destination is not None and poseFile is None:
            poseFile = open(destination,'w')
            if verbose:
                print("Writing pose data to {}".format(destination))
        # use visualization library
        engine3D = engine.Engine3js(grid=True, axis=True, view_width=windows_width, view_height=windows_height)

        if use_popup:
            # display the 3D human pose in this notebook, and origin frame in popup window
            display(engine3D.renderer)
            title = "Press ESC to Exit"
            cv2.namedWindow(title, cv2.WINDOW_KEEPRATIO | cv2.WINDOW_AUTOSIZE)
        else:
            # set the 2D image box, show both human pose and image in the notebook
            imgbox = widgets.Image(format="jpg", height=windows_height, width=windows_width)
            display(widgets.HBox([engine3D.renderer, imgbox]))

        skeleton = engine.Skeleton(body_edges=body_edges)

        processing_times = collections.deque()

        while True:
            # grab the frame
            frame = player.next()
            if frame is None:
                print("Source ended")
                break

            # resize image and change dims to fit neural network input
            # (see https://github.com/openvinotoolkit/open_model_zoo/tree/master/models/public/human-pose-estimation-3d-0001)
            scaled_img = cv2.resize(frame, dsize=(model.inputs[0].shape[3], model.inputs[0].shape[2]))

            if focal_length < 0:  # Focal length is unknown
                focal_length = np.float32(0.8 * scaled_img.shape[1])

            # inference start
            start_time = time.time()
            # get results
            inference_result = model_infer(scaled_img, stride)

            # inference stop
            stop_time = time.time()
            processing_times.append(stop_time - start_time)
            # Process the point to point coordinates of the data
            poses_3d, poses_2d = engine.parse_poses(inference_result, 1, stride, focal_length, True)

            # Print pose line once
            if not posePrinted and verbose:
                posePrinted = True
                print("Pose Data for this frame:")
                print(poses_2d)

            if exportPose:
                # Need to write a csv converter function for the pose data
                curStr = ""
                # curStr = curStr.append("{},".format(i)) for i in poses_2d[0]
                for i in poses_2d[0]:
                    curStr.append("{},".format(i))
                # poseFile.write(np.array2string(poses_2d))
                poseFile.write(curStr[:-1])
                poseFile.write("\n")
                
            # use processing times from last 200 frames
            if len(processing_times) > 200:
                processing_times.popleft()

            processing_time = np.mean(processing_times) * 1000
            fps = 1000 / processing_time

            if len(poses_3d) > 0:
                # From here, you can rotate the 3D point positions using the function "draw_poses",
                # or you can directly make the correct mapping below to properly display the object image on the screen
                poses_3d_copy = poses_3d.copy()
                x = poses_3d_copy[:, 0::4]
                y = poses_3d_copy[:, 1::4]
                z = poses_3d_copy[:, 2::4]
                poses_3d[:, 0::4], poses_3d[:, 1::4], poses_3d[:, 2::4] = (
                    -z + np.ones(poses_3d[:, 2::4].shape) * 200,
                    -y + np.ones(poses_3d[:, 2::4].shape) * 100,
                    -x,
                )

                poses_3d = poses_3d.reshape(poses_3d.shape[0], 19, -1)[:, :, 0:3]
                people = skeleton(poses_3d=poses_3d)

                try:
                    engine3D.scene_remove(skeleton_set)
                except Exception:
                    pass

                engine3D.scene_add(people)
                skeleton_set = people

                # draw 2D
                frame = draw_poses(frame, poses_2d, scaled_img, use_popup)

            else:
                try:
                    engine3D.scene_remove(skeleton_set)
                    skeleton_set = None
                except Exception:
                    pass

            if annotate:
                cv2.putText(
                    frame,
                    f"Inference time: {processing_time:.1f}ms ({fps:.1f} FPS)",
                    (10, 30),
                    cv2.FONT_HERSHEY_COMPLEX,
                    0.7,
                    (0, 0, 255),
                    1,
                    cv2.LINE_AA,
                )

            if use_popup:
                cv2.imshow(title, frame)
                if exportVideo:
                    videoOut.write(frame)
                key = cv2.waitKey(1)
                # escape = 27, use ESC to exit
                if key == 27:
                    break
            else:
                if exportVideo:
                    videoOut.write(frame)
                # encode numpy array to jpg
                imgbox.value = cv2.imencode(
                    ".jpg",
                    frame,
                    params=[cv2.IMWRITE_JPEG_QUALITY, 90],
                )[1].tobytes()

            engine3D.renderer.render(engine3D.scene, engine3D.cam)

    except KeyboardInterrupt:
        print("Interrupted")
    except RuntimeError as e:
        print(e)
    finally:
        if exportVideo:
            videoOut.release()
        if exportPose:
            poseFile.close()
        clear_output()
        if player is not None:
            # stop capturing
            player.stop()
        if use_popup:
            cv2.destroyAllWindows()
        if skeleton_set:
            engine3D.scene_remove(skeleton_set)

## Run
[back to top ⬆️](#Table-of-contents:)

Run, using a webcam as the video input. By default, the primary webcam is set with `source=0`. If you have multiple webcams, each one will be assigned a consecutive number starting at 0. Set `flip=True` when using a front-facing camera. Some web browsers, especially Mozilla Firefox, may cause flickering. If you experience flickering, set `use_popup=True`.

> **NOTE**:
>
> *1. To use this notebook with a webcam, you need to run the notebook on a computer with a webcam. If you run the notebook on a server (e.g. Binder), the webcam will not work.*
>
> *2. Popup mode may not work if you run this notebook on a remote computer (e.g. Binder).*

If you do not have a webcam, you can still run this demo with a video file. Any [format supported by OpenCV](https://docs.opencv.org/4.5.1/dd/d43/tutorial_py_video_display.html) will work.

Using the following method, you can click and move your mouse over the picture on the left to interact.

In [100]:
from notebook_utils import download_file
# import cv2
from cv2_enumerate_cameras import enumerate_cameras

camera_list = enumerate_cameras(cv2.CAP_DSHOW)
print(camera_list)

USE_WEBCAM = False

cam_id = 0
if not Path("face-demographics-walking.mp4").exists():
    download_file(
        "https://storage.openvinotoolkit.org/data/test_data/videos/face-demographics-walking.mp4",
    )
# video_path = "face-demographics-walking.mp4"
video_path = "Baden Test Screencap 1.mp4"
# video_path = "2025-09-20 Test Video 1.mp4"

source = cam_id if USE_WEBCAM else video_path

run_pose_estimation(source=source, flip=isinstance(source, int), use_popup=False,destination="testOut.csv",verbose=True,annotate=False,exportPose=True)

AttributeError: 'str' object has no attribute 'append'